## Importing the libraries

In [1]:
import numpy as np
import pandas as pd
import joblib

from rdkit_and_test_functions import smiles_to_3d,extract_3d_features

## Importing the dataset

In [2]:
dataset = pd.read_csv('CEPDB_25000.csv')

print("X column names:")
print(dataset.columns[1])
print("\ny column name:")
print(dataset.columns[8])

X column names:
SMILES_str

y column name:
e_gap_alpha


In [3]:
X_train = []
y_train = []

failed = 0

for _, row in dataset.iterrows():
    mol = smiles_to_3d(row["SMILES_str"])

    if mol is None:
        failed += 1
        continue

    try:
        features = extract_3d_features(mol)
        X_train.append(features)
        y_train.append(row["e_gap_alpha"])
    except Exception:
        failed += 1

# Convert list of dictionaries to numerical DataFrame
X_train = pd.DataFrame(X_train)
y_train = np.array(y_train, dtype=float)

print("X shape:", X_train.shape)
print("y shape:", y_train.shape)
print(X_train.head())
print(X_train.dtypes)

joblib.dump(X_train, "X_train.pkl")
joblib.dump(y_train, "y_train.pkl")

X shape: (24993, 11)
y shape: (24993,)
   RadiusOfGyration  Asphericity  Eccentricity  InertialShapeFactor  \
0          4.437465     0.579681      0.985711             0.000635   
1          4.760304     0.524671      0.980670             0.000417   
2          4.248867     0.658913      0.991408             0.000920   
3          4.180109     0.567643      0.984735             0.000714   
4          3.737606     0.422512      0.965632             0.000627   

   SpherocityIndex    MolWt  MolLogP   TPSA  HBD  HBA  RotBonds  
0     1.583060e-02  395.523   0.6943  67.35    1    5         1  
1     7.455893e-02  467.799   0.3595  15.79    1    2         1  
2     2.049153e-11  400.303   3.9315  67.35    1    4         1  
3     3.211014e-02  389.479   5.9806  26.03    0    3         1  
4     2.210589e-02  326.496   1.8308   0.00    0    1         0  
RadiusOfGyration       float64
Asphericity            float64
Eccentricity           float64
InertialShapeFactor    float64
SpherocityInde

['y_train.pkl']

In [3]:
X_train = joblib.load("X_train.pkl")
y_train = joblib.load("y_train.pkl")

print("Original X_train shape:", X_train.shape)
print("Original y_train shape:", y_train.shape)

hopv = pd.read_csv("HOPV_15_revised_2_processed_homo_5fold.csv")

# Take first 50 samples that have an experimental gap
hopv_50 = hopv.dropna(subset=["electrochemical_gap"]).head(50)

print("HOPV15 samples selected:", len(hopv_50))

X_hopv = []
y_hopv = []

failed = 0

for _, row in hopv_50.iterrows():

    mol = smiles_to_3d(row["smiles"])

    if mol is None:
        failed += 1
        continue

    try:
        features = extract_3d_features(mol)

        X_hopv.append(features)
        y_hopv.append(float(row["electrochemical_gap"]))

    except Exception:
        failed += 1

X_hopv = pd.DataFrame(X_hopv)
y_hopv = np.array(y_hopv, dtype=float)

X_hopv = X_hopv[X_train.columns]

X_train_modified = pd.concat(
    [X_train, X_hopv],
    ignore_index=True
)

y_train_modified = np.concatenate(
    [y_train, y_hopv]
)

joblib.dump(
    X_train_modified,
    "X_train_modified.pkl"
)

joblib.dump(
    y_train_modified,
    "y_train_modified.pkl"
)

print("\nHOPV15 successfully added:", len(X_hopv))
print("HOPV15 failed:", failed)

print("\nModified X_train shape:", X_train_modified.shape)
print("Modified y_train shape:", y_train_modified.shape)

print("\nSaved:")
print("X_train_modified.pkl")
print("y_train_modified.pkl")

Original X_train shape: (24993, 11)
Original y_train shape: (24993,)
HOPV15 samples selected: 50

HOPV15 successfully added: 50
HOPV15 failed: 0

Modified X_train shape: (25043, 11)
Modified y_train shape: (25043,)

Saved:
X_train_modified.pkl
y_train_modified.pkl
